# Task E — классификация приложений RuStore

Мультилейбл-классификация приложений с выдачей трёх наиболее вероятных рубрик.


## Зафиксированный результат

**Hitrate@3: 0.926** — лучший сохранённый авторский вариант задачи E.

> Метрика перенесена из авторского экспериментального ноутбука. Тяжёлые логи обучения удалены, чтобы решение хорошо отображалось на GitHub.


## Запуск

Положите данные в каталог `data/`, установите зависимости из корневого `requirements.txt` и последовательно выполните ячейки. Артефакты и сабмит будут записаны в `outputs/`.


In [ ]:
import os
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    AutoConfig,
    get_cosine_schedule_with_warmup
)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# ==========================================
# 0. Воспроизводимость (Seed)
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# Конфигурация
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LEN = 384          # 384 токена хватает на 99% сути описания
BATCH_SIZE = 16        # Если словите OOM (Out Of Memory) -> уменьшите до 8
EPOCHS = 4
LR = 3e-5
WEIGHT_DECAY = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Используем устройство: {DEVICE}")

# ==========================================
# 1. Загрузка и структурирование текста
# ==========================================
print("Загрузка данных...")
train_df = pd.read_csv(DATA_DIR / "train.tsv", sep="\t")
test_df = pd.read_csv(DATA_DIR / "test.tsv", sep="\t")

def format_text(df):
    """
    Трансформерам очень помогают префиксы, так они лучше разделяют важность полей.
    """
    app = df["app_name"].fillna("").astype(str)
    short = df["shortDescription"].fillna("").astype(str)
    full = df["full_description"].fillna("").astype(str)
    
    # Формируем структурированный промпт
    return "Название: " + app + " | Описание: " + short + " " + full

train_df["text"] = format_text(train_df)
test_df["text"] = format_text(test_df)

# Таргет
train_df["labels"] = train_df["labels_str"].str.split("|")
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df["labels"])
classes = list(mlb.classes_)
num_classes = len(classes)

# Сохраняем классы для инференса
np.save(OUTPUT_DIR / "classes.npy", classes)
print(f"Всего классов: {num_classes}")

# Разделение на train и val
train_texts, val_texts, y_tr, y_val, labels_tr, labels_val = train_test_split(
    train_df["text"].values, 
    y_train, 
    train_df["labels"].values, 
    test_size=0.15, 
    random_state=42
)

# ==========================================
# 2. PyTorch Dataset & DataLoader
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class RuStoreDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }
        
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
            
        return item

train_dataset = RuStoreDataset(train_texts, y_tr)
val_dataset = RuStoreDataset(val_texts, y_val)
test_dataset = RuStoreDataset(test_df["text"].values)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, pin_memory=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, pin_memory=True, num_workers=2)

# ==========================================
# 3. Модель и Оптимизация
# ==========================================
config = AutoConfig.from_pretrained(
    MODEL_NAME, 
    num_labels=num_classes,
    problem_type="multi_label_classification"
)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

# Дифференцированный Learning Rate (для классификатора чуть больше)
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {
        'params': [p for n, p in model.bert.named_parameters() if not any(nd in n for nd in no_decay)],
        'weight_decay': WEIGHT_DECAY,
        'lr': LR
    },
    {
        'params': [p for n, p in model.bert.named_parameters() if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0,
        'lr': LR
    },
    {
        'params': [p for n, p in model.classifier.named_parameters()],
        'weight_decay': WEIGHT_DECAY,
        'lr': LR * 3 # голова учится чуть быстрее
    }
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(total_steps * 0.1), 
    num_training_steps=total_steps
)
scaler = torch.cuda.amp.GradScaler() # Для FP16 ускорения

# ==========================================
# 4. Функция оценки (Hitrate@3)
# ==========================================
def evaluate(model, loader, true_labels):
    model.eval()
    all_probs = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            
            with torch.cuda.amp.autocast():
                logits = model(input_ids, attention_mask=attention_mask).logits
                probs = torch.sigmoid(logits)
                
            all_probs.append(probs.cpu().numpy())
            
    all_probs = np.vstack(all_probs)
    top3_idx = np.argsort(-all_probs, axis=1)[:, :3]
    top3_preds = [[classes[idx] for idx in row] for row in top3_idx]
    
    hits = sum(any(label in set(true) for label in pred) for true, pred in zip(true_labels, top3_preds))
    hitrate = hits / len(true_labels)
    return hitrate, all_probs

# ==========================================
# 5. Цикл обучения
# ==========================================
best_hitrate = 0.0
best_model_path = OUTPUT_DIR / "best_rubert_model.pt"

print("\n🚀 Начинаем обучение...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for batch in pbar:
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)
        
        with torch.cuda.amp.autocast():
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Защита от взрыва градиентов
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    avg_loss = running_loss / len(train_loader)
    val_hitrate, _ = evaluate(model, val_loader, labels_val)
    
    print(f"\n[Epoch {epoch}] Train Loss: {avg_loss:.4f} | Val Hitrate@3: {val_hitrate:.5f}")
    
    # Сохраняем лучший чекпоинт
    if val_hitrate > best_hitrate:
        best_hitrate = val_hitrate
        torch.save(model.state_dict(), best_model_path)
        print(f"🔥 Новая лучшая модель сохранена! (Hitrate@3 = {best_hitrate:.5f})")

# ==========================================
# 6. Инференс на тесте лучшей моделью
# ==========================================
print("\nЗагрузка лучшей модели для инференса...")
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_probs = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting Test"):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        
        with torch.cuda.amp.autocast():
            logits = model(input_ids, attention_mask=attention_mask).logits
            probs = torch.sigmoid(logits)
            
        test_probs.append(probs.cpu().numpy())

test_probs = np.vstack(test_probs)

# Сохраняем «сырые» вероятности для ансамбля
np.save(OUTPUT_DIR / "test_bert_probs.npy", test_probs)
print("Вероятности сохранены в 'test_bert_probs.npy'")

# Формируем сабмит
test_top3_idx = np.argsort(-test_probs, axis=1)[:, :3]
test_top3_preds = [[classes[idx] for idx in row] for row in test_top3_idx]
test_labels_str = ["|".join(preds) for preds in test_top3_preds]

submission = pd.DataFrame({
    "app_name": test_df["app_name"],
    "labels_str": test_labels_str
})

submission.to_csv(OUTPUT_DIR / "submission_bert.tsv", sep="\t", index=False)
print("Файл 'submission_bert.tsv' успешно создан!")
print(submission.head())
